# PANdeMaiz — Entrenamiento 1D-CNN para Detección Sísmica

**Propósito:** Entrenar una red neuronal liviana para clasificar ventanas de 4 s
como **Ruido (0)** o **Sismo (1)**, a partir de los espectrogramas generados en
`seismic_dataset_builder_v3.ipynb`.

| Parámetro | Valor |
|-----------|-------|
| Input shape | `(65, 11, 3)` float32 |
| Clases | 0 = Ruido (3 284 muestras) · 1 = Sismo (2 004 muestras) |
| Arquitectura | 1D-CNN (reshape frec→feature, Conv1D × 2) |
| Target tamaño | < 300 KB INT8 TFLite |
| Target recall sismo | ≥ 85 % en test set |
| PGA | No incluido en Fase 2 (el espectrograma lo codifica implícitamente) |

**Salidas generadas:**
- `ML/models/seismic_int8.tflite` — modelo cuantizado INT8 (pesos) / float32 (I/O)
- `ML/models/model.h` — arreglo C para incrustar en firmware ESP32
- `ML/models/norm_constants.h` — MEAN, STD y HANN_POWER para normalización en C++
- `ML/models/training_curves.png` — curvas de pérdida y AUC
- `ML/models/confusion_matrix.png` — matriz de confusión en test set


In [1]:
import os

# Forzar ejecución en CPU.
# Si tienes GPU y quieres usarla, comenta la siguiente línea:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")
print(f"GPUs visibles : {tf.config.list_physical_devices('GPU')}")
print(f"CPUs visibles : {tf.config.list_physical_devices('CPU')}")


I0000 00:00:1778679777.073419  202028 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow 2.21.0
GPUs visibles : []
CPUs visibles : [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Celda 2 — Imports y constantes

Centraliza todos los hiperparámetros del experimento en un único lugar para
facilitar la experimentación sin tocar el resto del notebook.

| Constante | Valor | Descripción |
|-----------|-------|-------------|
| `INPUT_SHAPE` | `(65, 12, 3)` | Forma de cada espectrograma (frec, tiempo, canal) |
| `BATCH_SIZE` | 32 | Muestras por mini-batch |
| `MAX_EPOCHS` | 100 | Límite superior; EarlyStopping cortará antes |
| `LR_INIT` | 1e-3 | Tasa de aprendizaje inicial de Adam |
| `SEED` | 42 | Semilla global para reproducibilidad del split |


In [2]:
import numpy as np
from pathlib import Path
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib
matplotlib.use('Agg')  # backend sin pantalla para nbconvert
import matplotlib.pyplot as plt

RUIDO_DIR   = Path("../Data_Labeling/Dataset/0_Ruido")
SISMO_DIR   = Path("../Data_Labeling/Dataset/1_Sismo")
MODEL_DIR   = Path("models"); MODEL_DIR.mkdir(exist_ok=True)

INPUT_SHAPE = (65, 11, 3)
BATCH_SIZE  = 32
MAX_EPOCHS  = 100
LR_INIT     = 1e-3
SEED        = 42


## Celda 4 — Carga del dataset

Carga todos los archivos `.npy` de ambas clases en arrays NumPy.
El orden de carga es reproducible (`sorted`) pero el split posterior mezcla
con semilla fija, por lo que el orden de carga no afecta el resultado.

**class_weight:** compensa el desbalance 1.64:1 (ruido:sismo) penalizando
más los errores sobre la clase minoritaria (sismo) durante el entrenamiento.
El factor es `n_ruido_train / n_sismo_train` para la clase 1; la clase 0 recibe peso 1.

**Split 70 / 15 / 15 estratificado:** mantiene la proporción 1.64:1 en
train, val y test, evitando que el test set tenga distribución diferente.


In [3]:
def load_dataset(ruido_dir, sismo_dir):
    X, y = [], []
    for npy in sorted(Path(ruido_dir).glob('*.npy')): X.append(np.load(npy)); y.append(0)
    for npy in sorted(Path(sismo_dir).glob('*.npy')): X.append(np.load(npy)); y.append(1)
    return np.stack(X, dtype=np.float32), np.array(y, dtype=np.int32)

X, y = load_dataset(RUIDO_DIR, SISMO_DIR)
print(f"Dataset cargado: {X.shape}  dtype={X.dtype}")
print(f"  Ruido (0): {np.sum(y==0)}   Sismo (1): {np.sum(y==1)}")

X_trv,  X_test,  y_trv,  y_test  = train_test_split(X,   y,   test_size=0.15,  random_state=SEED, stratify=y)
X_train, X_val,  y_train, y_val  = train_test_split(X_trv, y_trv, test_size=0.176, random_state=SEED, stratify=y_trv)

n0, n1 = int(np.sum(y_train==0)), int(np.sum(y_train==1))
class_weight = {0: 1.0, 1: round(n0 / n1, 4)}
print(f"\nTrain={len(y_train)}  Val={len(y_val)}  Test={len(y_test)}")
print(f"class_weight = {class_weight}")


Dataset cargado: (5288, 65, 11, 3)  dtype=float32
  Ruido (0): 3284   Sismo (1): 2004

Train=3703  Val=791  Test=794
class_weight = {0: 1.0, 1: 1.6393}


## Celda 6 — Normalización Z-score global

**¿Por qué normalizar?** Los valores raw de `log10(PSD + 1e-12)` oscilan
entre −12 (señal nula) y −2 (señal muy energética). Sin normalización,
la función de pérdida converge lento porque los gradientes son pequeños
fuera del rango [−1, 1].

**¿Por qué Z-score escalar (un solo MEAN y STD)?**
- Solo requiere 2 constantes en el firmware del ESP32 (2 floats = 8 bytes).
- El log-espectrograma tiene distribución aproximadamente normal, así que
  el Z-score global es casi tan efectivo como una normalización por bin.
- Se computa **únicamente sobre el training set** para evitar data leakage:
  si usáramos val o test para calcular la media, estaríamos "viendo" el
  futuro durante el entrenamiento.

**En el ESP32:** `x_norm = (log10(PSD + 1e-12) − MEAN) / STD`  
aplicado a cada uno de los 65 × 12 × 3 = 2 340 valores del espectrograma.


In [4]:
NORM_MEAN = float(X_train.mean())
NORM_STD  = float(X_train.std())
print(f"NORM_MEAN = {NORM_MEAN:.8f}")
print(f"NORM_STD  = {NORM_STD:.8f}")
print(f"Rango raw  : [{X_train.min():.2f}, {X_train.max():.2f}]")
X_n_tmp = (X_train - NORM_MEAN) / NORM_STD
print(f"Rango norm : [{X_n_tmp.min():.2f}, {X_n_tmp.max():.2f}]")
del X_n_tmp

X_train_n = (X_train - NORM_MEAN) / NORM_STD
X_val_n   = (X_val   - NORM_MEAN) / NORM_STD
X_test_n  = (X_test  - NORM_MEAN) / NORM_STD

np.savez(MODEL_DIR / "norm_params.npz",
         mean=np.float32(NORM_MEAN), std=np.float32(NORM_STD))
print("\nnorm_params.npz guardado")


NORM_MEAN = -9.83630657
NORM_STD  = 1.83801067
Rango raw  : [-12.00, -0.99]
Rango norm : [-1.18, 4.81]

norm_params.npz guardado


## Celda 8 — Arquitectura 1D-CNN

El espectrograma tiene forma `(65 frec, 11 tiempo, 3 canales)`.
Para una Conv1D necesitamos que la dimensión de **secuencia** sea el eje temporal:

```
(65, 11, 3)  →  Permute(2,1,3)  →  (11, 65, 3)  →  Reshape  →  (11, 195)
                                                                      ↑
                                               195 = 65 bins × 3 canales
```

Cada uno de los 11 pasos temporales (~0.33 s) tiene 195 features que
codifican el contenido frecuencial completo de los 3 ejes en ese instante.

| Capa | Salida | Parámetros |
|------|--------|------------|
| Input | (65, 11, 3) | — |
| Permute + Reshape | (11, 195) | 0 |
| Conv1D(32, k=3) + BN + ReLU | (11, 32) | 18 848 |
| MaxPool1D(2) | (5, 32) | 0 |
| Conv1D(64, k=3) + BN + ReLU | (5, 64) | 6 400 |
| MaxPool1D(2) | (2, 64) | 0 |
| GlobalAveragePooling1D | (64,) | 0 |
| Dropout(0.3) | (64,) | 0 |
| Dense(32, relu) | (32,) | 2 080 |
| Dense(1, sigmoid) | (1,) | 33 |
| **Total** | | **~27 500 params** |

FP32 ≈ 110 KB · INT8 ≈ 30 KB — bien por debajo del límite de 300 KB.


In [5]:
def build_model(input_shape=INPUT_SHAPE):
    inp = keras.Input(shape=input_shape)
    x   = keras.layers.Permute((2, 1, 3))(inp)        # (12, 65, 3)
    x   = keras.layers.Reshape((11, 65 * 3))(x)       # (11, 195)

    x = keras.layers.Conv1D(32, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.MaxPool1D(2)(x)                   # (5, 32)

    x = keras.layers.Conv1D(64, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.MaxPool1D(2)(x)                   # (2, 64)

    x = keras.layers.GlobalAveragePooling1D()(x)       # (64,)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out, name='seismic_1dcnn')

model = build_model()
model.summary()


Model: "seismic_1dcnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 65, 11, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 11, 65, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 11, 195)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 11, 32)         │        18,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 11, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 11, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 5, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 2, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,457 (107.25 KB)

 Trainable params: 27,265 (106.50 KB)

 Non-trainable params: 192 (768.00 B)

## Celda 10 — Entrenamiento

**Optimizador Adam:** adapta la tasa de aprendizaje por parámetro;
converge más rápido que SGD puro sin ajuste manual del scheduler.

**Métricas:**
- `accuracy` — orientativa (engañosa con clases desbalanceadas)
- `roc_auc` — área bajo la curva ROC; métrica principal de monitoreo

| Callback | Acción |
|----------|--------|
| `EarlyStopping(patience=15)` | Para si val_roc_auc no mejora 15 épocas |
| `ReduceLROnPlateau(patience=5)` | Divide LR × 0.3 si val_loss estanca |
| `ModelCheckpoint` | Guarda el mejor modelo según val_roc_auc |

`restore_best_weights=True` asegura que al final del entrenamiento el
modelo activo es el del mejor epoch, no el del último.


In [6]:
import tensorflow as tf

model.compile(
    optimizer=keras.optimizers.Adam(LR_INIT),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='roc_auc')]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_roc_auc', patience=15, mode='max', restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=5, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(
        MODEL_DIR / 'best_model.keras', monitor='val_roc_auc', mode='max', save_best_only=True)
]

history = model.fit(
    X_train_n, y_train,
    validation_data=(X_val_n, y_val),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set(title='Loss', xlabel='epoch'); ax1.legend()
ax2.plot(history.history['roc_auc'], label='train')
ax2.plot(history.history['val_roc_auc'], label='val')
ax2.set(title='ROC-AUC', xlabel='epoch'); ax2.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=100)
plt.close()
print("training_curves.png guardado")


Epoch 1/100


  1/116 ━━━━━━━━━━━━━━━━━━━━ 4:16 2s/step - accuracy: 0.5000 - loss: 1.0571 - roc_auc: 0.5875

 13/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5539 - loss: 0.9132 - roc_auc: 0.6583 

 29/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6206 - loss: 0.8370 - roc_auc: 0.6925

 44/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6486 - loss: 0.7989 - roc_auc: 0.7132

...

 87/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9328 - loss: 0.1983 - roc_auc: 0.9846

101/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9331 - loss: 0.1987 - roc_auc: 0.9845

115/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9335 - loss: 0.1991 - roc_auc: 0.9845

116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9379 - loss: 0.2009 - roc_auc: 0.9841 - val_accuracy: 0.8483 - val_loss: 0.3561 - val_roc_auc: 0.9188 - learning_rate: 1.0000e-06


training_curves.png guardado


## Celda 12 — Evaluación en test set

El test set (15% del total, ~794 muestras) no fue visto por el modelo
en ningún momento — ni para entrenar ni para ajustar hiperparámetros.

| Métrica | Objetivo | Razón |
|---------|----------|-------|
| Recall sismo | ≥ 0.85 | Un falso negativo (sismo no detectado) es inaceptable |
| F1 sismo | ≥ 0.80 | Balance precision/recall |
| Accuracy global | > 0.80 | Referencia; el recall sismo tiene prioridad |

Si el recall sismo cae por debajo de 0.80, considerar:
- Bajar el umbral de clasificación de 0.5 a 0.3 (tradeoff: más falsos positivos)
- Aumentar `class_weight[1]` y volver a entrenar
- Agregar PGA como feature adicional (Fase 2b)


In [7]:
y_prob = model.predict(X_test_n, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("=" * 55)
print(classification_report(y_test, y_pred,
                             target_names=['0_Ruido', '1_Sismo'], digits=4))
print("=" * 55)

disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                               display_labels=['Ruido', 'Sismo'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set (umbral = 0.5)')
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=100)
plt.close()
print("confusion_matrix.png guardado")


              precision    recall  f1-score   support

     0_Ruido     0.8843    0.8682    0.8762       493
     1_Sismo     0.7903    0.8140    0.8020       301

    accuracy                         0.8476       794
   macro avg     0.8373    0.8411    0.8391       794
weighted avg     0.8487    0.8476    0.8480       794

confusion_matrix.png guardado


## Celda 14 — Exportación a TFLite INT8

**¿Por qué INT8?**
- Los pesos FP32 (4 bytes/valor) se comprimen a INT8 (1 byte) → ~4× menos espacio.
- La inferencia en microcontroladores sin FPU dedicada es más rápida con enteros.
- El ESP32 (Xtensa LX6) no tiene unidad SIMD vectorial, pero TFLite Micro
  usa kernels de multiplicación optimizados para INT8.

**Configuración elegida: pesos INT8, I/O float32**

Se cuantizan los pesos y activaciones internas a INT8, pero la entrada y
salida del modelo permanecen en `float32`. Esto simplifica el firmware:
el ESP32 pasa directamente el array normalizado de floats sin necesitar
calcular manualmente los parámetros de cuantización de entrada
(`scale` y `zero_point`) que varían por capa.

**`representative_dataset`:** TFLite necesita 200–500 muestras reales del
training set para calibrar los rangos de activación de cada capa y asignar
los valores INT8 óptimos. Sin este paso, la cuantización sería estática
y perdería precisión.

| Archivo | Descripción |
|---------|-------------|
| `seismic_fp32.tflite` | Línea base sin cuantizar (referencia) |
| `seismic_int8.tflite` | Modelo listo para ESP32 (< 300 KB esperado) |


In [8]:
# ── FP32 baseline ─────────────────────────────────────────────────────
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32    = converter_fp32.convert()
(MODEL_DIR / 'seismic_fp32.tflite').write_bytes(tflite_fp32)
print(f"FP32 baseline : {len(tflite_fp32)/1024:.1f} KB")

# ── INT8 (pesos + activaciones), I/O permanece float32 ─────────────────
def repr_gen():
    """Muestras representativas para calibrar rangos de activación INT8."""
    for x in X_train_n[:300]:
        yield [x[np.newaxis].astype(np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations             = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset   = repr_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type     = tf.float32
converter_int8.inference_output_type    = tf.float32

tflite_int8 = converter_int8.convert()
(MODEL_DIR / 'seismic_int8.tflite').write_bytes(tflite_int8)
print(f"INT8 cuantizado: {len(tflite_int8)/1024:.1f} KB")
print(f"Compresión     : {len(tflite_fp32)/len(tflite_int8):.1f}×")

limite_kb = 300
ok = '✓' if len(tflite_int8)/1024 < limite_kb else '✗ SUPERA LÍMITE'
print(f"Objetivo < {limite_kb} KB : {ok}")


INFO:tensorflow:Assets written to: /tmp/tmpeg_4ijae/assets


INFO:tensorflow:Assets written to: /tmp/tmpeg_4ijae/assets


Saved artifact at '/tmp/tmpeg_4ijae'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 65, 11, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  129264525101008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525104272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525102928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525103888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525103120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525105424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525105616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525104656:

W0000 00:00:1778679815.479616  202028 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1778679815.479634  202028 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


I0000 00:00:1778679815.480462  202028 reader.cc:83] Reading SavedModel from: /tmp/tmpeg_4ijae
I0000 00:00:1778679815.481689  202028 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1778679815.481703  202028 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpeg_4ijae
I0000 00:00:1778679815.491564  202028 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1778679815.493235  202028 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1778679815.542936  202028 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpeg_4ijae
I0000 00:00:1778679815.560084  202028 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 79632 microseconds.
I0000 00:00:1778679815.627261  202028 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


...

INFO:tensorflow:Assets written to: /tmp/tmpoy3jqi9e/assets


Saved artifact at '/tmp/tmpoy3jqi9e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 65, 11, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  129264525101008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525104272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525102928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525103888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525101776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525103120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525105424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525105616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129264525104656:

/home/pinpa/Documents/Universidad/Trabajo_de_Grado_Seismology/PANdeMaiz/pan_env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 cuantizado: 40.1 KB
Compresión     : 2.9×
Objetivo < 300 KB : ✓


W0000 00:00:1778679816.485282  202028 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1778679816.485305  202028 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1778679816.485481  202028 reader.cc:83] Reading SavedModel from: /tmp/tmpoy3jqi9e
I0000 00:00:1778679816.486441  202028 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1778679816.486449  202028 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpoy3jqi9e
I0000 00:00:1778679816.497292  202028 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1778679816.541926  202028 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpoy3jqi9e
I0000 00:00:1778679816.560007  202028 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 74536 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1778679816.778197  202028 flatbuffer_expo

## Celda 16 — Cabeceras C para el firmware ESP32

Esta celda produce dos archivos `.h` listos para copiar a
`platform_acel/acel_ubuntu/src/`:

**`model.h`**  
Convierte `seismic_int8.tflite` a un arreglo C de bytes (`uint8_t[]`).
El ESP32 lo carga en flash (PROGMEM) y TFLite Micro lo interpreta en
tiempo de ejecución sin necesidad de sistema de archivos.

**`norm_constants.h`**  
Exporta las constantes de normalización calculadas en la Celda 7:
- `SEISMIC_NORM_MEAN` / `SEISMIC_NORM_STD` — para el Z-score en C++
- `SEISMIC_HANN_POWER` — suma de cuadrados de la ventana Hanning de 128
  puntos (idéntica a la usada por scipy), necesaria para que el PSD
  calculado en el ESP32 coincida numéricamente con el del entrenamiento.

> ⚠️ Si se re-entrena el modelo (nuevas épocas, más datos, hiperparámetros
> distintos), se deben regenerar y copiar **ambas** cabeceras. Las constantes
> de normalización cambian si el dataset de entrenamiento cambia.


In [9]:
import scipy.signal as signal

# Constante de ventana Hanning — idéntica a scipy.signal.get_window('hann', 128)
win        = signal.get_window('hann', 128)
HANN_POWER = float(np.sum(win ** 2))
print(f"HANN_POWER = {HANN_POWER:.8f}  (Σhann² de 128 puntos)")

# ── model.h ──────────────────────────────────────────────────────────────
data   = (MODEL_DIR / 'seismic_int8.tflite').read_bytes()
c_body = ', '.join(f'0x{b:02x}' for b in data)
(MODEL_DIR / 'model.h').write_text(
    f'#pragma once\n'
    f'// Auto-generado por seismic_cnn_trainer.ipynb — NO editar a mano\n'
    f'// Arquitectura : 1D-CNN  |  Cuantizacion : INT8 (I/O float32)\n'
    f'// Tamano       : {len(data)} bytes = {len(data)/1024:.1f} KB\n'
    f'// Input shape  : (1, 65, 12, 3) float32\n'
    f'// Output       : float32 score en [0=ruido, 1=sismo]\n'
    f'#include <cstdint>\n\n'
    f'const uint8_t  seismic_model_data[] = {{{c_body}}};\n'
    f'const uint32_t seismic_model_len    = {len(data)}u;\n',
    encoding='utf-8'
)
print(f"model.h generado ({len(data)/1024:.1f} KB)")

# ── norm_constants.h ─────────────────────────────────────────────────────
(MODEL_DIR / 'norm_constants.h').write_text(
    f'#pragma once\n'
    f'// Auto-generado por seismic_cnn_trainer.ipynb — NO editar a mano\n'
    f'// Aplicar en ESP32 ANTES de pasar el espectrograma al modelo:\n'
    f'//   x_norm = (log10(PSD + 1e-12) - MEAN) / STD\n\n'
    f'// Z-score global (calculado sobre el training set)\n'
    f'constexpr float SEISMIC_NORM_MEAN  = {NORM_MEAN:.8f}f;\n'
    f'constexpr float SEISMIC_NORM_STD   = {NORM_STD:.8f}f;\n\n'
    f'// Ventana Hanning 128 puntos — scipy.signal.get_window("hann", 128)\n'
    f'// PSD = |FFT|^2 / (FS * HANN_POWER)   (one-sided, igual que scipy)\n'
    f'constexpr float SEISMIC_HANN_POWER = {HANN_POWER:.8f}f;\n'
    f'constexpr float SEISMIC_FS         = 200.0f;\n\n'
    f'// Umbral de clasificacion (score > THRESHOLD -> sismo)\n'
    f'constexpr float SEISMIC_THRESHOLD  = 0.50f;\n',
    encoding='utf-8'
)
print("norm_constants.h generado")
print(f"\n-> Copiar ambos archivos a: platform_acel/acel_ubuntu/src/")


HANN_POWER = 48.00000000  (Σhann² de 128 puntos)
model.h generado (40.1 KB)
norm_constants.h generado

-> Copiar ambos archivos a: platform_acel/acel_ubuntu/src/
